## Compare the same models across both datasets and feature configurations

Run Notebook 03 first. Config A uses per-version features, Config B uses paired differences, and the combined configuration is retained as a baseline. Each configuration uses the same grouped cross-validation procedure on validated and significant data. Results go to matching `Results/validated/` and `Results/significant/` folders. Final energy-ready models default to Config B for validated data and combined features for significant data; change `FEATURE_CONFIG_BY_DATASET` in the first code cell to select another fit.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "Data" / "Processed").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run 03_feature_engineering.ipynb first to create Data/Processed.")
PROCESSED_DIR = ROOT / "Data" / "Processed"
RESULTS_DIR = ROOT / "Results" / "validated"
MODEL_DIR = PROCESSED_DIR / "models" / "validated"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df_ml_ready = pd.read_parquet(PROCESSED_DIR / "prepared_training_dataset.parquet")
manifest = json.loads((PROCESSED_DIR / "feature_manifest.json").read_text(encoding="utf-8"))
FEATURE_CONFIG_BY_DATASET = {"validated": "config_B", "significant": "combined"}
FEATURE_CONFIG = FEATURE_CONFIG_BY_DATASET["validated"]
feature_config_manifest_keys = {
    "config_A": "per_version",
    "config_B": "differences",
    "combined": "combined",
}
if FEATURE_CONFIG not in feature_config_manifest_keys:
    raise ValueError(f"Unknown feature configuration: {FEATURE_CONFIG}")
config_A_features = manifest["configurations"]["per_version"]
config_B_features = manifest["configurations"]["differences"]
clean_config_A_features = config_A_features
clean_config_B_features = config_B_features
# Materialize each matrix before selecting the configured one.
X_A_clean = df_ml_ready[clean_config_A_features].astype(float)
X_B_clean = df_ml_ready[clean_config_B_features].astype(float)
X_combined = df_ml_ready[manifest["configurations"]["combined"]].astype(float)
feature_matrices = {
    "config_A": X_A_clean,
    "config_B": X_B_clean,
    "combined": X_combined,
}
X = feature_matrices[FEATURE_CONFIG]
y_class = df_ml_ready["energy_winner"].astype(int)
groups = df_ml_ready["group_id"]
if groups.nunique() < 5:
    raise ValueError(f"GroupKFold(5) requires at least 5 problems; found {groups.nunique()}.")
if X.isna().any().any() or not np.isfinite(X.to_numpy()).all():
    raise ValueError("Feature matrix contains missing or non-finite values.")
if y_class.nunique() != 2:
    raise ValueError("Binary energy-winner training requires both target classes.")
gkf = GroupKFold(n_splits=5)
print(
    f"Loaded {len(df_ml_ready):,} non-tied pairs from {groups.nunique():,} problems; "
    f"configuration={FEATURE_CONFIG}, features={X.shape[1]}."
)


### Grouped cross-validation and final fitted models

All feature configurations use the same GroupKFold splits. Zero-variance features are identified using only each training fold and removed before fitting, preventing validation-fold information from affecting feature selection. Accuracy, balanced accuracy, F1, and ROC-AUC are reported.

In [ ]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiment_utils import train_tabular_configurations

model_factories = {
    "majority_baseline": lambda: DummyClassifier(strategy="most_frequent"),
    "logistic_regression": lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42),
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42, n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "xgboost": lambda: XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42,
        n_jobs=4, eval_metric="logloss",
    ),
}

feature_configurations = {
    "config_A": clean_config_A_features,
    "config_B": clean_config_B_features,
    "combined": manifest["configurations"]["combined"],
}
feature_config_descriptions = {
    "config_A": "per-version static features",
    "config_B": "paired feature differences (V1 - V0)",
    "combined": "Config A plus Config B",
}
performance, fitted_models, fitted_feature_columns = train_tabular_configurations(
    dataset_name="validated",
    frame=df_ml_ready,
    feature_configurations=feature_configurations,
    model_factories=model_factories,
    selected_feature_config=FEATURE_CONFIG_BY_DATASET["validated"],
    results_dir=RESULTS_DIR,
    model_dir=MODEL_DIR,
)
selected_columns = fitted_feature_columns["logistic_regression"]
(MODEL_DIR / "training_metadata.json").write_text(
    json.dumps({
        "dataset": "validated",
        "feature_config": FEATURE_CONFIG_BY_DATASET["validated"],
        "feature_config_description": feature_config_descriptions[FEATURE_CONFIG_BY_DATASET["validated"]],
        "feature_configurations_evaluated": feature_config_descriptions,
        "feature_columns": selected_columns,
        "target": "energy_winner",
        "group_column": "group_id",
        "group_kfold_splits": 5,
        "models": list(model_factories),
    }, indent=2),
    encoding="utf-8",
)
display(performance)
print(
    f"Saved all configuration fits under {MODEL_DIR}; "
    f"the selected {FEATURE_CONFIG_BY_DATASET['validated']} models are also available at the dataset root for energy measurement."
)


### Feature importance

Tree models use their fitted impurity/gain importances; Logistic Regression uses absolute standardized coefficients. These are descriptive full-training fits, while the reported performance remains GroupKFold out-of-fold performance.


In [ ]:
importance_tables = []
for model_name in ("logistic_regression", "random_forest", "xgboost"):
    estimator = fitted_models[model_name]
    if model_name == "logistic_regression":
        classifier = estimator[-1]
        importance = np.abs(classifier.coef_).mean(axis=0)
    else:
        importance = estimator.feature_importances_
    table = pd.DataFrame({"feature": fitted_feature_columns[model_name], "importance": importance})
    table["model"] = model_name
    importance_tables.append(table)

feature_importance = pd.concat(importance_tables, ignore_index=True)
feature_importance.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)
for model_name, table in feature_importance.groupby("model"):
    top_features = table.nlargest(20, "importance")[["feature", "importance"]].reset_index(drop=True)
    print(f"\nTop 20 features: {model_name}")
    print(top_features.to_string(index=False, formatters={"importance": "{:.4f}".format}))
    display(top_features)

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 8))
for ax, model_name in zip(axes, ("logistic_regression", "random_forest", "xgboost")):
    top = feature_importance[feature_importance["model"] == model_name].nlargest(15, "importance")
    ax.barh(top["feature"], top["importance"], color="steelblue")
    ax.invert_yaxis()
    ax.set(title=model_name.replace("_", " ").title(), xlabel="Importance", ylabel="Feature")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "feature_importance_top_features.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)



In [ ]:
dataset_name = "significant"
significant_results_dir = ROOT / "Results" / dataset_name
significant_model_dir = PROCESSED_DIR / "models" / dataset_name
significant_results_dir.mkdir(parents=True, exist_ok=True)
significant_model_dir.mkdir(parents=True, exist_ok=True)
df_significant_ml = pd.read_parquet(PROCESSED_DIR / "prepared_significant_training_dataset.parquet")
performance_significant, fitted_significant, significant_feature_columns = train_tabular_configurations(
    dataset_name=dataset_name,
    frame=df_significant_ml,
    feature_configurations=feature_configurations,
    model_factories=model_factories,
    selected_feature_config=FEATURE_CONFIG_BY_DATASET[dataset_name],
    results_dir=significant_results_dir,
    model_dir=significant_model_dir,
)
selected_significant_columns = significant_feature_columns["logistic_regression"]
(significant_model_dir / "training_metadata.json").write_text(json.dumps({
    "dataset": dataset_name,
    "feature_config": FEATURE_CONFIG_BY_DATASET[dataset_name],
    "feature_config_description": feature_config_descriptions[FEATURE_CONFIG_BY_DATASET[dataset_name]],
    "feature_configurations_evaluated": feature_config_descriptions,
    "feature_columns": selected_significant_columns,
    "target": "energy_winner",
    "group_column": "group_id",
    "group_kfold_splits": 5,
    "models": list(model_factories),
}, indent=2), encoding="utf-8")
display(performance_significant)
print(
    f"Saved all configuration fits under {significant_model_dir}; "
    f"the selected {FEATURE_CONFIG_BY_DATASET[dataset_name]} models are also available at the dataset root for energy measurement."
)


### CodeBERT embeddings on both datasets

Frozen `microsoft/codebert-base` mean-pools each code sample (256-token limit). A class-balanced Logistic Regression head uses both embeddings and their difference. The same five-fold problem-group evaluation runs for each dataset. The validated run is larger and takes longer. If needed, install once with `%pip install torch transformers safetensors`; the first run downloads CodeBERT.


In [ ]:
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from experiment_utils import train_codebert_dataset

DATASET_RUNS = {
    "validated": PROCESSED_DIR / "prepared_training_dataset.parquet",
    "significant": PROCESSED_DIR / "prepared_significant_training_dataset.parquet",
}
for dataset_name, dataset_path in DATASET_RUNS.items():
    train_codebert_dataset(
        dataset_name=dataset_name,
        dataset_path=dataset_path,
        results_dir=ROOT / "Results" / dataset_name,
        model_dir=PROCESSED_DIR / "models" / dataset_name,
    )
